## Multivariable Wing Leveler

In [1]:
import sys
sys.path.append('../')
from modeling.params_f16 import F16Params, Controls
from tools.lin_f16 import get_lin_f16 as linearize
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import tools.control_tools as tools

In [2]:
BETA_IDX = 0   # angle of ss
P_IDX = 1   # roll rate
R_IDX = 2   # yaw rate
PHI_IDX = 3 # bank angle

AIL_IDX = 0 # aileron control input
RUD_IDX = 1 # rudder control input

RTOD = 57.2958  # radians to degrees
DTOR = 0.0174533  # degrees to radians

In [3]:
params = F16Params()
params.alt_ft = 0.0
params.VT_ftps = 502.0
params.xcg = 0.35
controls = Controls()

In [4]:
_, lat_sys = linearize(controls, params)

Trim results:
Throttle (0-1): 0.26
Elevator (deg): -0.76
Alpha (deg): 2.12
Aileron (deg): 0.00
Rudder (deg): -0.00
Beta (deg): -0.00


In [5]:
KEEP = [BETA_IDX,P_IDX,R_IDX,PHI_IDX]
ap = lat_sys.A[KEEP][:, KEEP]
bp = lat_sys.B[KEEP]
cp = lat_sys.C[KEEP][:, KEEP]
dp = lat_sys.D[KEEP]
plant = ct.ss(ap, bp, cp, dp)
plant

<LinearIOSystem:sys[4]:['u[0]', 'u[1]']->['y[0]', 'y[1]', 'y[2]', 'y[3]']>

#### Cascade Actuators

In [10]:
aa = np.array([[-20.2, 0],[0, -20.2]])
ba = np.array([[20.2, 0],[0, 20.2]])
ca = np.eye(2)
da = np.zeros([2,2])
sysa = ct.ss(aa,ba,ca,da)
sys1 = ct.series(sysa, plant)
sys1

<LinearICSystem:sys[11]:['u[0]', 'u[1]']->['y[0]', 'y[1]', 'y[2]', 'y[3]']>

#### Cascade Washout

In [18]:
wt = 1
aw = -wt
bw = np.zeros([1,4])
bw[0,3] = 1
cw = np.zeros([5,1])
cw[4] = -wt
dw = np.eye(5,4)
dw[4,2] = 1
sysw = ct.ss(aw,bw,cw,dw)
sys2 = ct.series(sys1, sysw)
sys2[4,:]

StateSpace(array([[-2.02000000e+01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00, -2.02000000e+01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 2.94924368e-04,  8.05190337e-04, -3.21865180e-01,
         6.40399477e-02,  0.00000000e+00,  2.82226264e-02,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-7.33153314e-01,  1.31508944e-01, -3.06470738e+01,
         0.00000000e+00,  0.00000000e+00, -3.68259439e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  5.72957800e+01,
       

#### Generate Compensator